In [5]:
import pandas as pd
import numpy as np

# Rruga e saktë duke u futur te dosja processed
df_cleaned = pd.read_csv('../data/processed/berlin_airbnb_cleaned.csv') 

# Shiko dimensionet (sa rreshta dhe kolona ka tani)
print(f"Dataseti i pastruar ka {df_cleaned.shape[0]} rreshta dhe {df_cleaned.shape[1]} kolona.")
df_cleaned.head()
df_cleaned.info()

Dataseti i pastruar ka 4970 rreshta dhe 143 kolona.
<class 'pandas.DataFrame'>
RangeIndex: 4970 entries, 0 to 4969
Columns: 143 entries, Host Response Rate to Instant Bookable_t
dtypes: float64(14), int64(129)
memory usage: 5.4 MB


In [4]:
# Kontrollo llojin e të dhënave për çdo kolonë (numra apo tekst)
df_cleaned.info()

<class 'pandas.DataFrame'>
RangeIndex: 4999 entries, 0 to 4998
Data columns (total 46 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   index                  4999 non-null   int64  
 1   Review ID              4942 non-null   float64
 2   review_date            4942 non-null   str    
 3   Reviewer ID            4942 non-null   float64
 4   Reviewer Name          4942 non-null   str    
 5   Comments               4939 non-null   str    
 6   Listing ID             4999 non-null   int64  
 7   Listing URL            4999 non-null   str    
 8   Listing Name           4997 non-null   str    
 9   Host ID                4999 non-null   int64  
 10  Host URL               4999 non-null   str    
 11  Host Name              4998 non-null   str    
 12  Host Since             4998 non-null   str    
 13  Host Response Time     4366 non-null   str    
 14  Host Response Rate     4366 non-null   str    
 15  Is Superhost   

In [5]:
# Lista e kolonave që duam të fshijmë sepse nuk ndihmojnë në parashikimin e çmimit
columns_to_drop = [
    'index', 'Review ID', 'review_date', 'Reviewer ID', 'Reviewer Name', 
    'Comments', 'Listing ID', 'Listing URL', 'Listing Name', 'Host ID', 
    'Host URL', 'Host Name', 'Host Since', 'City', 'Postal Code'
]

# Fshijmë kolonat dhe krijojmë DataFrame-in e ri për Machine Learning
df_ml = df_cleaned.drop(columns=columns_to_drop, errors='ignore')

# Shiko dimensionet e reja dhe kolonat që mbetën
print(f"Dataseti i ri për Machine Learning ka {df_ml.shape[0]} rreshta dhe {df_ml.shape[1]} kolona.")
print("\nKolonat e mbetura:")
print(df_ml.columns.tolist())

Dataseti i ri për Machine Learning ka 4999 rreshta dhe 31 kolona.

Kolonat e mbetura:
['Host Response Time', 'Host Response Rate', 'Is Superhost', 'neighbourhood', 'Neighborhood Group', 'Country Code', 'Country', 'Latitude', 'Longitude', 'Is Exact Location', 'Property Type', 'Room Type', 'Accomodates', 'Bathrooms', 'Bedrooms', 'Beds', 'Price', 'Guests Included', 'Min Nights', 'Reviews', 'First Review', 'Last Review', 'Overall Rating', 'Accuracy Rating', 'Cleanliness Rating', 'Checkin Rating', 'Communication Rating', 'Location Rating', 'Value Rating', 'Instant Bookable', 'Business Travel Ready']


In [6]:
# Check main statistics for the price column
print("Price column descriptive statistics:")
print(df_ml['Price'].describe())

# Check the 5 highest prices to spot extreme outliers
print("\nTop 5 highest prices:")
print(df_ml['Price'].nlargest(5))

# Check for any zero or negative prices
print("\nNumber of rows where price is 0 or less:")
print((df_ml['Price'] <= 0).sum())

Price column descriptive statistics:
count    4999.000000
mean       68.352270
std        54.042753
min        10.000000
25%        37.000000
50%        53.000000
75%        80.000000
max       888.000000
Name: Price, dtype: float64

Top 5 highest prices:
1202    888.0
3711    580.0
2617    522.0
923     506.0
4981    506.0
Name: Price, dtype: float64

Number of rows where price is 0 or less:
0


In [7]:
# Check how many listings are above 300
expensive_listings = (df_ml['Price'] > 300).sum()
print(f"Number of listings with a price above 300: {expensive_listings}")

# Filter the dataset to keep only prices less than or equal to 300
df_ml = df_ml[df_ml['Price'] <= 300]

# Verify the new maximum price and rows left
print(f"\nNew dataset shape after removing outliers: {df_ml.shape}")
print(f"New maximum price: {df_ml['Price'].max()}")

Number of listings with a price above 300: 29

New dataset shape after removing outliers: (4970, 31)
New maximum price: 300.0


In [8]:
# Check which of the remaining columns have missing values
missing_counts = df_ml.isnull().sum()
missing_columns = missing_counts[missing_counts > 0].sort_values(ascending=False)

print("Remaining columns with missing values and their counts:")
if not missing_columns.empty:
    print(missing_columns)
else:
    print("Perfect! No missing values found in the remaining columns.")

Remaining columns with missing values and their counts:
Host Response Time      632
Host Response Rate      632
Cleanliness Rating       60
Location Rating          60
Accuracy Rating          60
Communication Rating     60
Checkin Rating           60
Value Rating             60
First Review             55
Last Review              55
Is Superhost              1
dtype: int64


In [12]:
# 1. Drop the date columns as they are not needed for ML algorithms
df_ml = df_ml.drop(columns=['First Review', 'Last Review'], errors='ignore')

# 2. Fill missing text/categorical values
df_ml['Host Response Time'] = df_ml['Host Response Time'].fillna('Unknown')
df_ml['Is Superhost'] = df_ml['Is Superhost'].fillna('False')

# List of numerical columns that we need to clean and fill
numerical_missing = [
    'Host Response Rate', 'Cleanliness Rating', 'Location Rating', 
    'Accuracy Rating', 'Communication Rating', 'Checkin Rating', 'Value Rating'
]

# 3. Force convert ALL these columns to numeric, then fill with median
for col in numerical_missing:
    # We clean the % sign by converting to string first, just in case
    df_ml[col] = df_ml[col].astype(str).str.replace('%', '', regex=False)
    
    # Force convert to numeric (any non-numeric text like 'nan' becomes NaN)
    df_ml[col] = pd.to_numeric(df_ml[col], errors='coerce')
    
    # Now calculate the median safely and fill the gaps
    median_value = df_ml[col].median()
    df_ml[col] = df_ml[col].fillna(median_value)

# 4. Final verification check
print("Remaining missing values count:")
print(df_ml.isnull().sum().sum())

Remaining missing values count:
0


In [14]:
# Select all categorical columns that are still text (object/string)
categorical_cols = df_ml.select_dtypes(include=['object', 'str']).columns.tolist()
print(f"Categorical columns to encode: {categorical_cols}")

# Apply One-Hot Encoding using pandas get_dummies
df_final_ml = pd.get_dummies(df_ml, columns=categorical_cols, drop_first=True, dtype=int)

# Print the final shape of our machine learning dataset
print(f"\nFinal dataset shape for Machine Learning: {df_final_ml.shape}")

Categorical columns to encode: ['Host Response Time', 'Is Superhost', 'neighbourhood', 'Neighborhood Group', 'Country Code', 'Country', 'Is Exact Location', 'Property Type', 'Room Type', 'Instant Bookable', 'Business Travel Ready']

Final dataset shape for Machine Learning: (4970, 143)


In [17]:
# 1. Select all categorical columns
categorical_cols = df_ml.select_dtypes(include=['object', 'str']).columns.tolist()
print(f"Categorical columns encoded: {categorical_cols}")

# 2. Apply One-Hot Encoding
df_final_ml = pd.get_dummies(df_ml, columns=categorical_cols, drop_first=True, dtype=int)
print(f"Final dataset shape for Machine Learning: {df_final_ml.shape}")

# 3. Save the final prepared dataset to the correct processed folder
output_path = '../data/processed/berlin_airbnb_cleaned.csv'
df_final_ml.to_csv(output_path, index=False)

print(f"\n[SUCCESS] The clean dataset has been saved to: {output_path}")

Categorical columns encoded: ['Host Response Time', 'Is Superhost', 'neighbourhood', 'Neighborhood Group', 'Country Code', 'Country', 'Is Exact Location', 'Property Type', 'Room Type', 'Instant Bookable', 'Business Travel Ready']
Final dataset shape for Machine Learning: (4970, 143)

[SUCCESS] The clean dataset has been saved to: ../data/processed/berlin_airbnb_cleaned.csv
